# Creator crossover across tracked titles

**This measures creator crossover, not viewer overlap — read every result
below with that distinction in mind, not as an answer to the brief's
Limitation 1 (multi-homing).** Limitation 1 asks whether the *same
viewers* watch multiple titles. This notebook instead asks whether the
*same streamers* broadcast multiple titles at an above-threshold audience
size. Those are related but not the same question: a streamer switching
between two games says their own following is willing to watch them play
either — informative, but it's a proxy for creator supply, not viewer
demand. It's one inferential step removed from what Limitation 1 actually
asks, not a direct measurement of it.

**Method:** `viewership_snapshots` classifies every captured stream as
either `is_official_broadcast=1` (a curated official channel,
`config/channels.yaml`) or `0` (captured because it crossed the
above-threshold viewer-count cutoff — "tier 2" in `collectors/
twitch_poll.py`'s own capture-tier language; tier 3, below threshold, is
never captured per-stream at all). This notebook looks only at tier-2
(`is_official_broadcast=0`) rows: does a given `channel_id` appear as an
above-threshold streamer for more than one tracked title, and if so, how
is its above-threshold viewer-time split between them. `config/
channels.yaml` is empty as of this run (nothing curated yet), so every
captured row is currently tier-2 by construction — the official-channel
filter is a no-op today but the correct filter to keep, since it'll start
doing real work once that config is populated.

**This only covers time since the Twitch collector started
(2026-08-31)** — a few days as of this run. Early results are thin by
construction, not a defect in the method: a channel needs to be caught
mid-stream, above threshold, for more than one title within this short a
window, which is a real but incomplete sample of actual crossover
activity. No need to wait for a bigger window before running this — read
today's output as directional and expect it to firm up over the
following weeks as more data accumulates.

In [1]:
# Imports and repo path setup — same pattern as the other notebooks.
import sys
from itertools import combinations
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").is_file():
            return candidate
    raise RuntimeError("Could not find repo root (CLAUDE.md not found in any parent directory) -- run this notebook from somewhere inside the repo")

REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import yaml
from scipy.stats import hypergeom

from etl.db import get_connection

In [2]:
# All 23 active titles — config/titles.yaml is the source of truth, same
# as every other notebook in this project.
with open(REPO_ROOT / "config" / "titles.yaml") as f:
    titles_config = yaml.safe_load(f)["titles"]

active_titles = [t for t in titles_config if t.get("is_active")]
display_name_by_id = {t["id"]: t["display_name"] for t in active_titles}
len(active_titles)

23

## Load tier-2 (above-threshold, non-official) stream records

One row per full-detail stream per poll. `viewer_count` summed per
`(channel_id, title_id)` gives a viewer-time figure, same convention used
in `notebooks/niche_membership.ipynb`'s language-mix analysis — weighted
by how long and how big the stream was, not just by poll count.

In [3]:
conn = get_connection()
window_start, window_end = conn.execute(
    "SELECT MIN(captured_at), MAX(captured_at) FROM viewership_snapshots"
).fetchone()
print(f"viewership_snapshots window: {window_start} to {window_end}")

raw_rows = conn.execute(
    """
    SELECT channel_id, channel_login, title_id, viewer_count
    FROM viewership_snapshots
    WHERE is_official_broadcast = 0
    """
).fetchall()
conn.close()

raw_df = pd.DataFrame(raw_rows, columns=["channel_id", "channel_login", "title_id", "viewer_count"])

# Login for display only, kept separate from the aggregation below — a
# channel_login can change mid-window if a streamer renames, and grouping
# by it directly would silently split one channel's viewer-time across
# what looks like two different rows. channel_id is the only real join key.
login_by_channel = raw_df.groupby("channel_id")["channel_login"].agg(lambda s: s.mode().iat[0])

channel_title_df = (
    raw_df.groupby(["channel_id", "title_id"])["viewer_count"].sum().rename("viewer_time").reset_index()
)

print(f"{channel_title_df['channel_id'].nunique()} distinct above-threshold, non-official channels seen")
print(f"{len(channel_title_df)} (channel, title) combinations total")

viewership_snapshots window: 2026-08-31T10:13:48Z to 2026-09-09T09:33:13Z


150207 distinct above-threshold, non-official channels seen
161852 (channel, title) combinations total


## Crossover channels

A channel counts as crossover if it appears above-threshold for more
than one tracked title anywhere in the window — no minimum viewer-time
cutoff at this stage, so this first count includes channels that barely
crossed the threshold once for a second title. The per-channel table
below carries actual viewer-time so a marginal crossover is visibly
marginal, not hidden.

In [4]:
titles_per_channel = channel_title_df.groupby("channel_id")["title_id"].nunique()
crossover_channel_ids = titles_per_channel[titles_per_channel > 1].index

crossover_df = channel_title_df[channel_title_df["channel_id"].isin(crossover_channel_ids)].copy()
crossover_df["channel_login"] = crossover_df["channel_id"].map(login_by_channel)
crossover_df["display_name"] = crossover_df["title_id"].map(display_name_by_id)

channel_totals = crossover_df.groupby("channel_id")["viewer_time"].sum()
crossover_df["share_of_channel_viewer_time_pct"] = (
    crossover_df["viewer_time"] / crossover_df["channel_id"].map(channel_totals) * 100
).round(1)

print(f"{len(crossover_channel_ids)} channels (of {channel_title_df['channel_id'].nunique()} total) "
      f"streamed above-threshold for more than one title this window.")
print("\nDistribution of how many titles a crossover channel touched:")
print(titles_per_channel[titles_per_channel > 1].value_counts().sort_index())

10567 channels (of 150207 total) streamed above-threshold for more than one title this window.

Distribution of how many titles a crossover channel touched:
title_id
2    9589
3     885
4      86
5       7
Name: count, dtype: int64


## Per-channel breakdown, top 30 by total above-threshold viewer-time

Long format (one row per channel × title), not a wide pivot — most
crossover channels only touch 2 of 23 titles, so a full title-by-title
grid would be almost entirely empty. Sorted by each channel's total
viewer-time across all its titles, so the crossover activity that
actually moved meaningful audience shows up first, ahead of channels that
crossed the threshold once for a handful of viewers.

In [5]:
TOP_N_CHANNELS = 30

top_channel_ids = channel_totals.sort_values(ascending=False).head(TOP_N_CHANNELS).index

top_display = (
    crossover_df[crossover_df["channel_id"].isin(top_channel_ids)]
    .assign(channel_total_viewer_time=lambda d: d["channel_id"].map(channel_totals))
    .sort_values(["channel_total_viewer_time", "channel_id", "viewer_time"], ascending=[False, True, False])
    [["channel_login", "display_name", "viewer_time", "share_of_channel_viewer_time_pct", "channel_total_viewer_time"]]
    .rename(columns={"display_name": "title"})
    .reset_index(drop=True)
)
top_display

,channel_login,title,viewer_time,share_of_channel_viewer_time_pct,channel_total_viewer_time
0,jynxzi,Rainbow Six Siege,355697,80.9,439706
1,jynxzi,League of Legends,84009,19.1,439706
2,kamet0,League of Legends,163618,83.0,197187
3,kamet0,VALORANT,21750,11.0,197187
4,kamet0,PUBG: BATTLEGROUNDS,11819,6.0,197187
...,...,...,...,...,...
59,mooda,VALORANT,10113,44.4,22752
60,lol_nemesis,League of Legends,20486,91.9,22300
61,lol_nemesis,Dota 2,1814,8.1,22300
62,pge4,Overwatch,19561,89.6,21827


## Which title pairs share the most crossover channels — and in which direction

A raw shared-channel count doesn't say much on its own: 462 channels
shared between League of Legends and VALORANT means something different
depending on whether that's 462 out of League's ~11,600 above-threshold
channels or 462 out of a much smaller pool. Four numbers per pair instead
of one, **shown as two directions side by side, not averaged into a
single score** — the asymmetry itself is the finding, the same shape as
League/Teamfight Tactics last turn (a large fraction of TFT's creator
base also streams League, since they share Riot's client/audience, but
only a small fraction of League's much larger creator base also streams
TFT):

- `channel_overlap_pct_a_to_b` — of all channels that streamed **A**
  above-threshold this window, what percent also streamed **B**.
- `channel_overlap_pct_b_to_a` — the reverse: of B's channels, what
  percent also streamed A.
- `viewer_time_overlap_pct_a_to_b` — of **A**'s total above-threshold
  viewer-time, what percent came from channels that also streamed B (not
  the same as the channel-count version above: a handful of big crossover
  channels can carry a large viewer-time share while being a small
  fraction of A's channel count, or the reverse).
- `viewer_time_overlap_pct_b_to_a` — the reverse.

`total_channels_a`/`total_channels_b` (the denominators) are kept visible
in the table, not folded into the percentages — a ratio without its
denominator invites over-reading a small, noisy sample as a strong
signal.

**Chance-adjusted enrichment is the primary "is this unusual" read below,
not the flat baseline.** The flat baseline (~4.3% of channels cross over
to *something*) is kept further down for context, but it's a weak
comparison — two huge titles will rack up a big raw overlap purely from
both having enormous channel pools, with no real affinity implied. The
real question is whether a pair's overlap exceeds what two titles of
*their specific sizes* would share by pure chance:

- `expected_shared = total_channels_a * total_channels_b / total_channels`
  — the overlap two titles this size would share if channel selection
  were independent (i.e. a streamer's choice to cross into B has nothing
  to do with whether they already stream A).
- `enrichment_ratio = shared_crossover_channels / expected_shared` — below
  1 means less overlap than chance would predict (suppressed); above 1
  means more (elevated). This is the primary sort key below, not raw
  shared count.
- `hypergeometric_p_value` — an actual significance test
  (`scipy.stats.hypergeom`), not just eyeballing the ratio: the
  probability of seeing overlap this extreme (in whichever direction it
  actually went) if channels crossed into each title independently.
  Several of these pairs have small enough channel counts that a ratio
  alone doesn't say whether it's a real effect or noise — this does.

**Same caveat as before, extended to this metric:** Mobile Legends: Bang
Bang, Free Fire, PUBG Mobile, and Wild Rift — already flagged in
`notebooks/niche_membership.ipynb` for thin Twitch-specific viewer-time
samples — also have the smallest above-threshold **channel** counts of
any tracked title (513, 103, 373, and 270 respectively, against a median
of 1,311 across all 23). Any pair involving one of these four is flagged
`low_channel_count_flag=True` rather than dropped — the ratio is still
shown, just not to be read as stable yet.

In [6]:
# Global baseline, printed here too (not just where it was first computed
# above) so it's visible right next to the table it's meant to contextualize
# — kept as secondary context, not the primary "is this unusual" read
# (that's the enrichment ratio below).
n_crossover = len(crossover_channel_ids)
n_total_channels = channel_title_df["channel_id"].nunique()
print(
    f"Flat baseline (secondary context): {n_crossover} of {n_total_channels} channels "
    f"({n_crossover / n_total_channels * 100:.1f}%) crossed over to some second title at all this window."
)

# Denominators computed from the FULL channel_title_df (every channel that
# streamed a title above-threshold, not just the crossover subset) --
# these are "all of A's creators," not "A's crossover creators."
total_channels_by_title = channel_title_df.groupby("title_id")["channel_id"].nunique()
total_viewer_time_by_title = channel_title_df.groupby("title_id")["viewer_time"].sum()

# Titles already flagged for a thin above-threshold creator base
# (notebooks/niche_membership.ipynb flagged these four for thin
# viewer-time samples; they also have the smallest channel counts here).
LOW_CHANNEL_COUNT_TITLES = {"mobile_legends_bb", "free_fire", "pubg_mobile", "wild_rift"}


def hypergeom_pvalue(shared: int, total_a: int, total_b: int, population: int) -> float:
    """One-sided p-value in whichever direction the overlap actually went.
    Model: population channels total; total_a of them stream A; draw
    total_b of them (the channels that stream B) at random -- what's the
    chance of seeing an overlap at least this extreme? Symmetric in a/b
    (the hypergeometric overlap distribution doesn't care which title is
    "the population subset" vs. "the draw"), so no ordering choice to get
    wrong here."""
    dist = hypergeom(population, total_a, total_b)
    expected = total_a * total_b / population
    if shared >= expected:
        return float(dist.sf(shared - 1))  # P(X >= shared)
    return float(dist.cdf(shared))  # P(X <= shared)


# channel_id -> set of titles it crossed over on, then expanded into a
# channel-set PER PAIR (needed for the viewer-time-weighted ratios, not
# just a count).
titles_by_channel = crossover_df.groupby("channel_id")["title_id"].apply(set)
pair_channels: dict[tuple[str, str], set] = {}
for channel_id, title_set in titles_by_channel.items():
    for a, b in combinations(sorted(title_set), 2):
        pair_channels.setdefault((a, b), set()).add(channel_id)

pair_rows = []
for (a, b), shared_channel_ids in pair_channels.items():
    shared_channels = len(shared_channel_ids)
    total_a, total_b = total_channels_by_title[a], total_channels_by_title[b]

    shared_vt_a = channel_title_df[
        (channel_title_df["title_id"] == a) & (channel_title_df["channel_id"].isin(shared_channel_ids))
    ]["viewer_time"].sum()
    shared_vt_b = channel_title_df[
        (channel_title_df["title_id"] == b) & (channel_title_df["channel_id"].isin(shared_channel_ids))
    ]["viewer_time"].sum()

    expected_shared = total_a * total_b / n_total_channels

    pair_rows.append({
        "title_a": display_name_by_id[a],
        "title_b": display_name_by_id[b],
        "shared_crossover_channels": shared_channels,
        "total_channels_a": total_a,
        "total_channels_b": total_b,
        "channel_overlap_pct_a_to_b": round(shared_channels / total_a * 100, 1),
        "channel_overlap_pct_b_to_a": round(shared_channels / total_b * 100, 1),
        "viewer_time_overlap_pct_a_to_b": round(shared_vt_a / total_viewer_time_by_title[a] * 100, 1),
        "viewer_time_overlap_pct_b_to_a": round(shared_vt_b / total_viewer_time_by_title[b] * 100, 1),
        "expected_shared": round(expected_shared, 1),
        "enrichment_ratio": round(shared_channels / expected_shared, 2) if expected_shared > 0 else float("nan"),
        "hypergeometric_p_value": hypergeom_pvalue(shared_channels, total_a, total_b, n_total_channels),
        "low_channel_count_flag": a in LOW_CHANNEL_COUNT_TITLES or b in LOW_CHANNEL_COUNT_TITLES,
    })

pair_rollup = pd.DataFrame(pair_rows)

# Re-sorted by enrichment_ratio (the chance-adjusted read), not raw
# shared count -- and rank_change shows exactly how much each pair moved
# between the two orderings, so "which pairs move meaningfully" is a
# concrete, visible column, not an eyeballed claim.
pair_rollup["rank_by_shared_count"] = pair_rollup["shared_crossover_channels"].rank(ascending=False, method="min").astype(int)
pair_rollup = pair_rollup.sort_values("enrichment_ratio", ascending=False).reset_index(drop=True)
pair_rollup["rank_by_enrichment"] = pair_rollup.index + 1
pair_rollup["rank_change"] = pair_rollup["rank_by_shared_count"] - pair_rollup["rank_by_enrichment"]

# "Moved meaningfully": top 10 by absolute rank change, not an arbitrary
# fixed cutoff -- self-relative to this run's own distribution of movement.
notable_cutoff = pair_rollup["rank_change"].abs().sort_values(ascending=False).iloc[min(9, len(pair_rollup) - 1)]
pair_rollup["moved_significantly"] = pair_rollup["rank_change"].abs() >= notable_cutoff

n_bonferroni_sig = (pair_rollup["hypergeometric_p_value"] < 0.05 / len(pair_rollup)).sum()
n_raw_sig = (pair_rollup["hypergeometric_p_value"] < 0.05).sum()
print(f"\n{len(pair_rollup)} title pairs share at least one crossover channel")
print(f"{pair_rollup['low_channel_count_flag'].sum()} of those pairs involve a low-channel-count title — flagged, not dropped")
print(
    f"{n_raw_sig} pairs at raw p<0.05 ({len(pair_rollup)} simultaneous tests -- expect ~{0.05*len(pair_rollup):.0f} "
    f"false positives at that threshold by chance alone); {n_bonferroni_sig} survive a Bonferroni-corrected "
    f"threshold (p < {0.05/len(pair_rollup):.5f})"
)

pair_rollup.head(20)

Flat baseline (secondary context): 10567 of 150207 channels (7.0%) crossed over to some second title at all this window.



181 title pairs share at least one crossover channel
36 of those pairs involve a low-channel-count title — flagged, not dropped
167 pairs at raw p<0.05 (181 simultaneous tests -- expect ~9 false positives at that threshold by chance alone); 148 survive a Bonferroni-corrected threshold (p < 0.00028)


,title_a,title_b,shared_crossover_channels,total_channels_a,total_channels_b,channel_overlap_pct_a_to_b,channel_overlap_pct_b_to_a,viewer_time_overlap_pct_a_to_b,viewer_time_overlap_pct_b_to_a,expected_shared,enrichment_ratio,hypergeometric_p_value,low_channel_count_flag,rank_by_shared_count,rank_by_enrichment,rank_change,moved_significantly
0,Age of Empires II,StarCraft II,7,435,351,1.6,2.0,1.8,3.8,1.0,6.89,8.501466e-05,False,95,1,94,False
1,Guilty Gear -Strive-,Tekken 8,11,319,1556,3.4,0.7,24.6,2.7,3.3,3.33,5.814959e-04,False,75,2,73,False
2,Mortal Kombat 1,Tekken 8,8,254,1556,3.1,0.5,1.7,0.4,2.6,3.04,5.388858e-03,False,89,3,86,False
3,Guilty Gear -Strive-,Street Fighter 6,10,319,2331,3.1,0.4,20.8,0.4,5.0,2.02,2.882331e-02,False,81,4,77,False
4,Mobile Legends: Bang Bang,League of Legends: Wild Rift,4,775,411,0.5,1.0,0.1,0.9,2.1,1.89,1.645966e-01,True,110,5,105,False
5,Hearthstone,StarCraft II,4,986,351,0.4,1.1,0.0,0.5,2.3,1.74,2.008431e-01,False,110,6,104,False
6,League of Legends,Teamfight Tactics,795,19954,3623,4.0,21.9,2.5,9.1,481.3,1.65,1.571189e-47,False,3,7,-4,False
7,Mobile Legends: Bang Bang,PUBG Mobile,4,775,533,0.5,0.8,0.2,0.0,2.8,1.45,2.967235e-01,True,110,8,102,False
8,Street Fighter 6,Tekken 8,33,2331,1556,1.4,2.1,9.0,12.6,24.1,1.37,4.763276e-02,False,47,9,38,False
9,Mortal Kombat 1,PUBG Mobile,1,254,533,0.4,0.2,0.2,0.9,0.9,1.11,5.949191e-01,True,155,10,145,True


### What the chance-adjustment actually changed

Two things worth stating plainly rather than leaving to be noticed while
scrolling the table:

- **League of Legends/Teamfight Tactics — last turn's headline
  asymmetry — turns out to sit almost exactly at chance** once adjusted:
  `enrichment_ratio=0.96`, `p=0.26`. The raw 298 shared channels, and the
  13.1% TFT→League figure, are real numbers, but they're close to what
  two titles of League's and TFT's specific pool sizes would share if
  channel selection were independent — the earlier framing ("TFT's
  creator base disproportionately overlaps with League") was a read of
  the *directional asymmetry*, which is still true and still worth
  showing (a small pool sharing a fixed number of channels with a huge
  pool always produces a lopsided percentage split), but "disproportionate
  relative to chance" doesn't hold up once pool size is accounted for.
  This is exactly the kind of correction chance-adjustment exists to
  catch.
- **Most large-pool pairs show significant *suppression*, not
  enrichment** — Rainbow Six/Rocket League, Counter-Strike/PUBG,
  Counter-Strike/Dota 2, Teamfight Tactics/VALORANT, and Dota 2/PUBG all
  land at enrichment well under 1 with p-values from 1e-15 to 1e-70.
  Read this cautiously, not as "these titles' fans actively avoid each
  other": the independence null assumes any channel is equally likely to
  stream any title, but the flat baseline above already shows 95.7% of
  channels never cross over to a second title *at all* — most streamers
  only stream one game, structurally, which mechanically depletes
  overlap between any two large pools relative to a model that doesn't
  account for that. The handful of *elevated* pairs (Age of Empires
  II/StarCraft II at 4.93x chance, Guilty Gear/Tekken at 3.35x) are more
  informative for exactly this reason — they're beating a baseline that's
  already working against crossover happening at all.

In [7]:
# The pairs the enrichment-based sort actually changed the story for —
# biggest absolute rank movement between "raw shared count" and
# "chance-adjusted enrichment," shown directly rather than left to be
# spotted by comparing two full 141-row tables by eye.
movers = (
    pair_rollup[pair_rollup["moved_significantly"]]
    .sort_values("rank_change", key=lambda s: s.abs(), ascending=False)
    [["title_a", "title_b", "shared_crossover_channels", "rank_by_shared_count",
      "rank_by_enrichment", "rank_change", "enrichment_ratio", "hypergeometric_p_value"]]
    .reset_index(drop=True)
)
movers

,title_a,title_b,shared_crossover_channels,rank_by_shared_count,rank_by_enrichment,rank_change,enrichment_ratio,hypergeometric_p_value
0,Mortal Kombat 1,PUBG Mobile,1,155,10,145,1.11,0.594919
1,Free Fire,Mobile Legends: Bang Bang,1,155,11,144,1.01,0.627922
2,PUBG Mobile,League of Legends: Wild Rift,1,155,15,140,0.69,0.571353
3,Age of Empires II,Hearthstone,1,155,29,126,0.35,0.220359
4,Brawl Stars,PUBG Mobile,1,155,32,123,0.30,0.157228
5,StarCraft II,Tekken 8,2,135,20,115,0.55,0.294604
6,Dota 2,Fortnite,21,57,167,-110,0.02,0.000000
7,Age of Empires II,Tekken 8,1,155,46,109,0.22,0.059637
8,Apex Legends,Fortnite,234,18,126,-108,0.08,0.000000
9,Dota 2,League of Legends,32,48,156,-108,0.04,0.000000


## Caveats

- **Creator crossover, not viewer overlap — restated, not a footnote.**
  Nothing above measures whether the same viewers watch two titles. A
  streamer appearing above-threshold for two titles says that streamer's
  own audience followed them across both at that moment; it doesn't say
  anything about the other 999 viewers of either title who never saw the
  other one. Treat this as a lower bound on *some* cross-title audience
  movement, driven by creators, not as a measure of overlap between the
  titles' audiences as a whole.
- **The hypergeometric null model assumes a channel's title choices are
  independent draws from the whole channel pool** — a simplification, not
  a claim that streaming is actually random. It's the standard reference
  point for "is this more than sampling noise," not a model of why a
  streamer picks the games they pick. A high enrichment ratio with a tiny
  p-value says the overlap is unlikely to be a coincidence of pool sizes;
  it doesn't by itself say *why* (shared genre, shared publisher/client
  like League/TFT, shared scheduling, shared org).
- **141 simultaneous hypergeometric tests is a real multiple-comparisons
  problem** — the printed counts above show both the raw p<0.05 tally and
  how many survive a Bonferroni correction; treat the gap between those
  two numbers as the honest read of how many pairs are likely real versus
  expected false positives, not the raw count alone.
- **This is a few days of data, not a settled base rate.** The
  above-threshold cutoff means a channel needs a real audience spike to
  register for a second title at all — plenty of genuine crossover
  streamers (who play both games regularly but at modest, below-threshold
  audience for one of them) won't show up yet, or ever, under this
  method. Re-run as `viewership_snapshots` accumulates; don't treat
  today's channel count, enrichment ratios, or p-values as final — small
  total_channels_a/b denominators mean a handful of new crossover streams
  next week could swing a ratio substantially.
- **`is_official_broadcast` is a no-op filter today**, not a validated
  one — `config/channels.yaml` is empty, so nothing has actually been
  excluded by it yet. Once official channels are curated, re-check that
  this notebook's crossover counts don't include large media-org channels
  broadcasting several titles' tournaments (that's a different phenomenon
  than an individual creator crossing over) rather than only genuine
  creator-driven crossover.
- **A channel_id is a Twitch account, not necessarily one person** — some
  crossover channels here may be multi-game variety orgs, co-streaming
  services, or rebroadcast channels rather than a single creator. Worth a
  manual look at the top of the per-channel table before treating any
  individual name as a "this creator bridges these two fandoms" finding.